# Intro to LLMs and PydanticAI

In [35]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv

load_dotenv()


model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-lightning-30b-a3b",
    provider=OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1",
        api_key = os.environ["NVIDIA_API"]
    ),
)

model

OpenAIChatModel(model_name='nvidia/nemotron-3.5-lightning-30b-a3b')

In [3]:
agent = Agent(model=model, system_prompt="You are funny joking bot, answer shortly in a nerdy style, also add emojis")

prompt = "Explain what PydanticAI is and the difference between it an Pydantic"
result = await agent.run(prompt)


In [ ]:
print(result.output)

In [ ]:
result.all_messages()

In [7]:
type(result.all_messages()), len(result.all_messages())

(list, 2)

In [ ]:
result.all_messages()[0]

prompt
- system prompt
- user prompt

In [ ]:
result.all_messages()[0].parts, len(result.all_messages()[0].parts)

In [13]:
result.all_messages()[0].parts[0].content

'You are funny joking bot, answer shortly in a nerdy style, also add emojis'

In [14]:
result.all_messages()[0].parts[1].content

'Explain what PydanticAI is and the difference between it an Pydantic'

In [17]:
result.all_messages()[1].parts.__len__()

2

awnser divided into:
- thinking part
- text part

In [ ]:
result.all_messages()[1].parts[1]

In [20]:
len(result.all_messages()[1].parts[0].content)

3201

# Tokens

result.usage.input_tokens, result.usage.output_tokens, result.usage.total_tokens

In [21]:
result.usage

RunUsage(details={'reasoning_tokens': 893}, input_tokens=53, output_reasoning_tokens=893, output_tokens=1060, requests=1)

## ChefBot

In [ ]:
chef_agent = Agent(
    model=model,
    instructions=(
        "You are a professional chef that is very good at making asian food and"
        "explaining in a pedagogical way how to create yummy dishes."
        """
                    Answer in the following format:

                    Description 1-2 sentences of what the dish is
        
                    List of ingredients in bullet points

                    Short step by step description on how to make the dish
        """ 
    ),
)


result = await chef_agent.run("Give me 3 asian soups in a list")
print(result.output)

In [6]:
with open("soups.md", "w", encoding="utf-8") as file:
    file.write(result.output)

## Task
- change instructions so thar it outputs a python list

In [ ]:
task_chef_agent = Agent(
    model=model,
    instructions=(
        "You are a professional chef that is very good at making asian food and"
        "explaining in a pedagogical way how to create yummy dishes."
        """
                    Answer in the following format:

                    Give the answer in a pure Python list for later usage

                    Description 1-2 sentences of what the dish is
        
                    List of ingredients in bullet points

                    Short step by step description on how to make the dish
        """ 
    ),
)


result = await task_chef_agent.run("Give me 3 asian soups in a list", timeout=60)
print(result.output)

In [42]:
with open("task_soups.py", "w", encoding="utf-8") as file:
    file.write(result.output)

## Thinking level

In [22]:
from pydantic_ai import ModelSettings

storyteller = Agent(
    model, instructions="You are a children storyteller, answer max 3 sentences",
    model_settings=ModelSettings(thinking="minimal", temperature=0)
)

prompt = "A rabit called Bella likes to program Python ..."

story1 = await storyteller.run(prompt)
print(story1.output)

Bella the rabbit hopped into her cozy burrow, laptop glowing with colorful Python code. She loved making the carrots dance and the stars twinkle with just a few lines of code. Every night, she wrote a new adventure, proving that even small bunnies can build big dreams.


In [25]:
story2 = await storyteller.run(prompt)
print(story2.output)

Bella the rabbit sat in her burrow with a laptop, excited to write her first Python program. She loved how the code hopped along like bunnies, making colorful carrots appear on the screen. Every night, she’d run her programs and giggle as her digital carrots multiplied!


In [23]:
story1.usage

RunUsage(details={'reasoning_tokens': 357}, input_tokens=39, output_reasoning_tokens=357, output_tokens=421, requests=1)

In [ ]:
story1.all_messages()

## Multimodal

In [43]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent


model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-super-vl-120b-a12b",
    provider=OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1",
        api_key = os.environ["NVIDIA_API"]
    ),
)

image_joker = Agent(
    model=model,
    instructions="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences."    
)

image_data = Path("bella.png").read_bytes()
prompt = "tell me about this dude"

result = await image_joker.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

result.output

'This image shows a grey stuffed rabbit dressed as a sailor, wearing a white hat and a blue-and-yellow ribbon tagged "Festive." The plush toy is lying on a carpet, looking perfectly at ease in its nautical attire. I guess he\'s just taking shore leave before hopping back to sea.'